In [3]:
!pip install datasets
!pip install loralib
!pip install trl
!pip install accelerate
!pip install transformers

In [ ]:
# git clone https://github.com/airobotlab/KoChatGPT
# cp -r  ~/Projects/KoChatGPT/colossalai_ChatGPT_230319/chatgpt ~/Projects/content/chatgpt

In [1]:
import os
import sys

# 1. 홈 디렉토리 경로 자동 변환 + 중간에 겹친 chatgpt/chatgpt 구조 반영
HOME = os.path.expanduser("~")
BASE_PATH = os.path.join(HOME, "Projects/content")
GPT_PATH =  f"{BASE_PATH}/chatgpt"

if BASE_PATH not in sys.path:
    sys.path.append(BASE_PATH)

In [ ]:
modifications = [
    {
        "file": f"{GPT_PATH}/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": f"{GPT_PATH}/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": f"{GPT_PATH}/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": f"{GPT_PATH}/trainer/base.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": f"{GPT_PATH}/trainer/rm.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])


✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/trainer/callbacks/save_checkpoint.py
✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/trainer/strategies/__init__.py
✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/dataset/reward_dataset.py
✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/trainer/base.py
✅ 수정 완료: /Users/jamesyang/Projects/content/chatgpt/trainer/rm.py


In [2]:
import torch
import transformers
# AutoTokenizer가 한국어 한글의 유니코드(UTF-8) 바이트를 제대로 조합하지 못하고 문자열을 산산조각
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import PreTrainedTokenizerFast
import pandas as pd
import numpy
import json

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))



# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

Torch version:2.10.0
Cuda version: None
transformers version: 5.1.0
GPU 사용 가능여부: False


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

# SKT의 한국어 생성 모델(KoGPT2) pretraied와  토큰나이저
model_name = "skt/kogpt2-base-v2"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2",
  bos_token="</s>", eos_token="</s>", unk_token="<unk>",
  pad_token="<pad>", mask_token="<mask>")
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# 토크나이저 한 번에 자를 수 있는 문장의 최대 길이(토큰 개수)
tokenizer.model_max_length

1000000000000000019884624838656

In [5]:
#  모델은 1번부터 1024번까지만 위치 번호표를 만들 수 있게 짓자
model.config.n_positions

1024

In [6]:
tokenizer.model_max_length = model.config.n_positions # 토크나이저 최대 길이를 모델 뇌 용량 1024로 고정!

In [7]:
# 우리가 자연스럽게 읽는 한국어 문장을 "인공지능 모델이 읽을 수 있는 숫자 배열(ID)"로 어떻게 변환시키는지를 한눈에 확인하는 과정입니다!

input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."
tokens = tokenizer(input_txt).tokens()
print(tokens)
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()

['▁바람', '도', '▁없는', '▁공중에', '▁수직', '의', '▁파', '문을', '▁내', '이며', '▁고', '요', '히', '▁떨어지는', '▁오동', '잎은', '▁누', '구의', '▁발자', '취', '▁입', '니까', '.']


In [8]:
pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kogpt-2_tokens", "Input_IDs"])
df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
kogpt-2_tokens,▁바람,도,▁없는,▁공중에,▁수직,의,▁파,문을,▁내,이며,▁고,요,히,▁떨어지는,▁오동,잎은,▁누,구의,▁발자,취,▁입,니까,.
Input_IDs,10891,7235,9712,49207,14438,8143,9203,9941,9094,9639,9065,8084,8811,21215,34769,19985,9669,10139,21626,8408,9241,23775,389


In [9]:
# 생성되는 전체 문장(질문 포함)의 길이가 최대 128개 조각(토큰)
max_length=128

input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
# 드디어 AI 모델이 일을 시작합니다. 방금 변환한 입력값을 주고 .generate() 함수로 뒷말을 지어내라고 명령합니다.
# do_sample=False: AI의 상상력을 통제하는 아주 중요한 스위치입니다. 
# False로 꺼두면, AI는 매번 다음 단어를 고를 때 무조건 자기가 배운 것 중 "가장 뻔하고 정답일 확률이 1등으로 가장 높은 단어" 한 개만 기계적으로 고르게 됩니다. (이 방식을 욕심쟁이처럼 가장 확률 높은 것만 집어먹는다고 해서 **'그리디 탐색(Greedy Search)'**이라고 부릅니다.)
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
print(tokenizer.decode(output_greedy[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇다면 그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리요?"
"그건 무슨 소리


In [10]:
# "여러 갈래의 미래를 동시에 계산(Beam Search)해서 가장 완벽하고 매끄러운 문장"을 지어내도록 인공지능에게 지시하는 똑똑한 문장 생성 코드입니다.

input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)

# num_beams=10 (빔 탐색 10갈래)

# no_repeat_ngram_size=2 (똑같은 말 반복 금지)
# 언어 모델들이 가장 자주 하는 실수인 "그래 그래 그래 그래", "나는 나는 밥을 밥을" 같은 말더듬이 버그를 원천 차단하는 옵션입니다.
# "2개 단어(2-gram)가 똑같이 연달아 2번 반복되면 무조건 오답 처리해라!"라는 엄격한 규칙입니다.

# do_sample=False 랜덤성은 끄고, 수학적으로 가장 완벽한(확률 높은) 길만 걷도록 합니다.
output_beam = model.generate(input_ids, max_length=max_length, num_beams=10, no_repeat_ngram_size=2,
                             do_sample=False)
print(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
"그렇지 않습니다."
"어떻게 된 일입니까?"
그녀는 고개를 갸웃거렸다.
"아니, 그게 무슨 말씀이신지 모르겠습니다만."
"무슨 말씀인지 알 수가 없군요."
아무런 대답도 하지 않은 채 그녀는 고개를 끄덕였다.
"그래, 알았어."
그녀의 눈에서 눈물이 주르륵 흘러내렸다.
그녀가 다시 입을 열었다.
"정말 죄송합니다, 고마워요, 고맙습니다"
"


In [11]:
# do_sample=True (창의력 스위치 ON!)
# 이전에는 이게 False로 꺼져 있어서 무조건 확률이 가장 높은(가장 뻔한) 1등 단어만 골랐습니다.
# 방금 이 스위치를 **True**로 켰기 때문에, 이제 모델은 무조건 1등 단어만 고집하지 않고
#  "가끔은 2등이나 3등 단어도 섞어서 문장을 이어가 볼까? 그래야 좀 더 사람 같고 창의적이니까!"라며 
#  약간의 주사위 굴리기(Sampling)를 시작합니다. 똑같은 코드를 두 번 실행하면 매번 다른 대답이 나오게 됩니다.

# top_k=50 (이상한 헛소리 방지!)
# 상상력을 켜주었더니, 가끔 모델이 확률 꼴등(10만 등)인 완전 엉뚱한 외계어 단어를 주사위로 뽑아버리는 대참사가 일어날 수 있습니다.
# 그래서 "아무리 상상력을 발휘해도, 무조건 상위 50등(Top 50) 안에 드는 말이 되는 단어들 중에서만 주사위를 굴려!"라고 
# 안전망을 쳐주는 옵션입니다.

# temperature=2.0
# 이 숫자는 "얼마나 평범함을 거부할 것인가?"를 조절하는 온도 값입니다. (보통 0.1 ~ 2.0 사이를 씁니다.)
# temperature가 0.1 일 때 (차가움): 아주 조심스럽고 보수적입니다. 거의 1등 단어만 뽑는 차가운 로봇 같습니다.
# temperature가 2.0 일 때 (뜨거움 / 약간 미쳤음): 1등 단어와 50등 단어를 뽑을 확률을 거의 비등비등하게 평준화시켜버립니다.
# 아주 과감하게 모험을 떠나서 정말 예측할 수 없는 특이한 문장이나 아주 어색한 헛소리가 나올 확률이 몹시 높아집니다!


output_beam = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, temperature=2.0, top_k=50)
print(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까.'
어둠이 드리우기 시작하는 새벽녘, 나는 창밖을 내다보았다.
순간 나는 문득 가슴이 설레었다.
창밖은 벌써 어둠의 도가니로 변해 있었다.
밤이 깊어가는 시각은 한낮이었다.
나는 창문을 두드리며 창문 밖을 내려다보았다.
밤바람이 차갑게 불기 시작했다.
나는 어둠에 휩싸여 있는 나의 모습을 가만히 지켜보았다.
그런데 그 어둠이 어느새 내 가슴 속을 채우고 있었다.
나는 더 이상 견디기 힘든 지경에 이르렀다.
그렇다고 해도


In [12]:
# 누클리어스 샘플링(Nucleus Sampling)"**이라고 불리는 가장 현대적이고 똑똑한 단어 선택 방식(top_p)을 사용해 문장을 지어내는 코드입니다.
# 1. 이전 방식(top_k=50)의 치명적인 단점
# 어제 배웠던 top_k=50은 "무조건 상위 50등까지만 보고 골라라!"라는 무식한 방법이었습니다.

# 상황 A: 정답이 아주 명확할 때 ("대한민국의 수도는 __"). 1등(서울)이 정답일 확률이 99%인데 굳이 50등까지 이상한 단어를 살펴볼 필요가 없습니다. 하지만 top_k는 억지로 50등까지의 쓰레기 단어들을 후보에 올려버려서 가끔 헛소리가 나옵니다.
# 상황 B: 정답이 불확실할 때 ("어제 점심으로 __"). 피자, 햄버거, 짜장면 등 다양한 단어가 올 수 있는데 50등까지만 자르면 너무 뻔한 단어만 나와서 상상력이 부족해집니다.
# 2. 구세주 등장: top_p=0.90 (유동적인 컷오프)
# 이 옵션은 등수(k)로 자르는 게 아니라 **"쓸만한 단어들의 확률을 위에서부터 더해서 총합이 90%(0.90)가 될 때까지만 후보를 추려라!"**라는 아주 똑똑한 규칙입니다.

# 정답이 뻔할 때 (예: 서울 90%, 부산 2% ...): 1등 단어(서울) 딱 하나만 담아도 벌써 90%가 차버리니까 후보 상표를 1개로 확 닫아버립니다. (헛소리 완벽 차단!)
# 정답이 여러 개일 때 (예: 피자 10%, 햄버거 10% ...): 90%를 채우려면 상위 20~30개의 단어를 잔뜩 담아야 합니다. (알아서 후보를 넓혀 상상력을 발휘!)

output_beam = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, top_p=0.90)
print(tokenizer.decode(output_beam[0]))

바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."
"그렇군. 그게 무슨 소리예요?"
그녀는 한숨을 내쉬었다.
"무슨 말인지 알겠어요. 그건 그렇고."
"아무것도 아니잖아요. 아무 것도 아니란 말이에요."
나는 고개를 끄덕였다.
"이런, 저런저런 말씀을 하시는 거예요. 어젯밤에도 말씀하셨지요. 아까도 말씀드렸습니다만."
그녀가 내뱉은 말이었다.
"어떻게 된 거죠


In [13]:
# "대화형 AI(ChatGPT)를 만들기 위해 사람들이 미리 준비해 둔 '정답 모음집(학습 데이터)' 
# 파일을 파이썬으로 불러와서 읽어 들이는 과정"**입니다.
# 보통 3단계(SFT 👉 RM 👉 PPO)
# SFT 단계

import json
data_path_1_SFT = '/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl'
with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '불고기용 고기 한우에요?',
  'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
  'tokens': 193},
 {'prompt': '쓰던 앱이 유료로 전환됐어',
  'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
  'tokens': 288},
 {'prompt': '여친이랑 다툼',
  'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
  'tokens': 153}]

In [14]:
# 방금 전 코드는 3단계(PPO) 모의고사를 치를 때 옆에서 점수를 매겨줄 "자동 채점기(RM)"를 똑똑하게 훈련시키기 위해, 
# 사람들이 정성껏 등수를 매겨둔 10,220개의 채점 기준판 데이터를 불러오는 과정입니다!


data_path_2_RM = '/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl'
with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

10220


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
  'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
  'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
  'completion_2': '라이언에게 말했다.',
  'ranking': [2, 1, 0]},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
  'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
  'completion_1': '이날 목송에서 구글상위노',
  'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
  'ranking': [2, 0, 1]},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
  'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
  'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문

In [15]:
# "강화학습(PPO)"을 위해 쓸 특별한 시험지 파일을 꺼내오는 과정입가
# kochatgpt_3_PPO 데이터셋의 Prompt는 kochatgpt_2_RM 데이터셋에서 사용된 질문(prompt)들을 기반으로 하였으며 
# RLHF에서 정책 모델(Policy Model) 을 학습시키기 위해 사용되었습니다.

data_path_3_PPO = '/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl'
with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

12000


[{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?'},
 {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?'},
 {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?'}]

In [7]:
from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy

In [5]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
# tokenizer = AutoTokenizer.from_pretrained(
#     'skt/kogpt2-base-v2', bos_token='</s>', eos_token='</s>', unk_token='</s>', pad_token='</s>',
#     padding_side="right",
#     model_max_length=512,
# )

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token='</s>', 
    eos_token='</s>', 
    unk_token='<unk>',  # 변경 (모르는 단어)
    pad_token='<pad>',  # 변경 (빈칸 채우기용 특수기호)
    mask_token='<mask>', # 추가 
    padding_side="right",
    model_max_length=512
)
print(tokenizer)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TokenizersBackend(name_or_path='skt/kogpt2-base-v2', vocab_size=51200, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<usr>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<sys>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<mask>", rstrip=False, lstrip=False, single_word=False, normalized=False,

In [6]:
# 1. 테스트할 한글 문장을 준비합니다.
test_text = "안녕하세요. 이것은 깨짐 확인 테스트입니다."
# 2. 토크나이저 가위로 잘라봅니다.
tokens = tokenizer.tokenize(test_text)
# 3. 어떻게 잘렸는지 눈으로 확인합니다.
print("자른 결과물 👉:", tokens)

자른 결과물 👉: ['▁안녕', '하', '세', '요.', '▁이것은', '▁깨', '짐', '▁확인', '▁테', '스트', '입니다.']


In [7]:
# [{'prompt': '불고기용 고기 한우에요?',
#   'completion': "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.",
#   'tokens': 193},
#  {'prompt': '쓰던 앱이 유료로 전환됐어',
#   'completion': "'어떤 앱인지 모르기 때문에 정확한 답변을 드리기 어렵습니다. 하지만, 일반적으로 유료 전환된 앱은 기존 무료 앱에서 추가적인 기능이나 서비스를 제공하는 경우가 많습니다. 이 경우, 유료 전환 이전 가격이 매우 저렴하거나 무료인 경우가 많으므로, 앱을 계속 사용하려면 추가적인 비용을 지불해야 할 수 있습니다. 하지만, 유료 전환 이후에는 기존 사용자 또는 새로운 사용자 모두가 추가 기능과 높은 품질의 서비스를 이용할 수 있게 됩니다. 따라서, 앱 개발자는 유료 모델로 전환함으로써 앱의 수익을 증가시키고 더 나은 서비스를 제공할 수 있습니다.",
#   'tokens': 288},
#  {'prompt': '여친이랑 다툼',
#   'completion': "'저는 인공지능이기 때문에 여친과의 다툼에 대해서는 조언을 드릴 수는 없습니다. 하지만 다투게 되면 상대방의 말을 경청하는 태도와 서로의 감정을 존중하는 것이 중요합니다. 논리적이고 차분한 대화를 통해 문제를 해결해보세요. 그리고 서로를 이해하고 서로의 의견을 수용하는 것이 중요합니다.",
#   'tokens': 153}]



class SFT_dataset(Dataset):
    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")

        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response

        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)

        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }

        prompt_input = PROMPT_DICT["prompt_input"]

        #source는 질문...
        sources = []
        for example in list_data_dict:
            tmp = prompt_input.format_map(example)
            sources.append(tmp)

        # targets은 정답리스트
        targets = []
        for example in list_data_dict:
            # 딕셔너리에서 "completion(정답)" 내용만 꺼낸 뒤, 
             # 맨 끝자락에 인공지능이 "나 말 다 했어!"라고 알려주는 특수기호(eos_token)인 '</s>'를 강제로 붙여줍니다.
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")

        # zip()으로 금방 만들었던 두 바구니(sources, targets)를 나란히 세워놓고,
        # s(질문) 한 덩어리, t(정답) 한 덩어리를 차례대로 꺼내서 서로 딱 달라붙게(+) 더해줍니다!
        examples = [s + t for s, t in zip(sources, targets)]

#  첫 번째 바구니 (질문 포장지)
# sources = [
#     "### Instruction(명령어):\n불고기용 고기 한우에요?\n\n### Response(응답):\n",
#     "### Instruction(명령어):\n마디반지 세트로 구매할 수 있나요?\n\n### Response(응답):\n"
# ]
# # 두 번째 바구니 (정답 내용 + 마침표)
# targets = [
#     "저는 인공지능 챗봇이며...</s>",
#     "네, 저희 쇼핑몰에서는...</s>"
# ]

# 이 코드가 실행된 후 examples 안의 모습:
# [
#     # 첫 번째 완성본 (examples[0])
#     "### Instruction(명령어):\n불고기용 고기 한우에요?\n\n### Response(응답):\n저는 인공지능 챗봇이며...</s>",
    
#     # 두 번째 완성본 (examples[1])
#     "### Instruction(명령어):\n마디반지 세트로 구매할 수 있나요?\n\n### Response(응답):\n네, 저희 쇼핑몰에서는...</s>"
# ]

        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target

        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids)

        # **sources_tokenized["input_ids_lens"]**는 정말 직관적이고 중요한 정보를 담고 있는 바구니입니다. 
        # 바로 **"각각의 질문(Source) 문장들이 몇 개의 단어(토큰) 조각으로 잘려 있는지 그 '길이(Length) 숫자 모음'"**입니다.
        
        # (CrossEntropyLoss 함수)에는 아주 특별한 룰이 하나 있습니다. 
        # "만약 정답지(Label)에 써진 숫자가 -100이라면, 그 단어는 아예 투명 취급하고 감점(Loss)을 주지 마라!" 
        # (이를 전문 용어로 ignore_index라고 부릅니다.)

        # "질문:" 뒤에 "사과의"가 와야 한다고 예측하는 것은 중요할까요? 아니요! 질문은 어차피 사용자가 던지는 내용이니까 AI가 굳이 맞춰야 할 필요가 없습니다. 저걸 맞추라고 감점(Loss)을 주면, AI가 쓸데없이 질문 외우기에 뇌 용량을 다 써버립니다 😭
        # 그래서 채점관용 **정답지 앞부분 6칸(label[:6])을 -100으로 지워놓고, 
        # 오직 "답변:" 뒤에 나오는 진짜 인공지능의 말투(나머지 6칸)만 틀렸을 때 호되게 감점(채점)**을 하겠다는 아주 지능적인 전략입니다.
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100

        data_dict = dict(input_ids=input_ids, labels=labels)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))


    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt", # 파이토치(PyTorch)가 좋아하는 텐서 형태로 바꿔줘!
                padding="longest", # 짧은 문장들은 제일 긴 문장 길이에 맞춰 빈칸(<pad>)을 채워줘!
                max_length=tokenizer.model_max_length, # 모델 뇌 용량(1024)을 넘어가면 무조건 싹둑 잘라! (truncation=True)
                truncation=True,
            )
            for text in strings  # 우리가 넘겨준 1만 2천 개의 문장(strings)을 하나씩 꺼내면서 반복해.
        ]
        # 토크나이저 가위를 거치고 나면 잡다한 포장지가 많이 붙어있습니다. 이 코드는 포장지를 다 벗겨내고, 
        # 진짜 알맹이인 '단어 번호표 배열([1024, 381, 99...])'만 쏙 꺼내서 input_ids라는 이름의 리스트에 차곡차곡 담는 과정입니다.
        #  (나중에 정답지로도 복사해서 쓸 거라 labels에도 같은 값으로 이름을 하나 더 달아주었습니다.)
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )


    def __len__(self):
        return len(self.input_ids)


    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [8]:
# 길이가 제각각인 문장들을 모아서, 가장 긴 것에 맞춰 빈칸을 채우고(pad_sequence), 
# 빈칸은 채점하지 않도록 설정하며(-100), 모델이 어디가 진짜 글자인지 알 수 있게 지도(ttention_mask)를 그려준다


@dataclass
class DataCollatorForSupervisedDataset(object):

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

In [12]:
train_dataset = SFT_dataset(data_path_1_SFT='/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl', tokenizer=tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])

input : tensor([  739,   378,   378,   378, 14659, 13394, 37091, 10651,   383, 25841,
         8006, 14914,   375,  7673, 20479,  8091, 22311,  9036, 30902, 13675,
          375,   378,   378,   378, 41951,   454,  9549, 20549,   383,  8142,
         7192, 14914,   382, 37767, 13753,  8263,  7166,   739,  8352,  7659,
         9594, 25585, 13600,  8022,  9378, 11532,  9887, 11218,  9111, 16691,
        10351, 10561,  9128, 20479,  8091,  9065,  9446,  9036, 28420, 26521,
        10163, 26367,  6958,  9030,  9882, 12317, 25882,  9209, 37194, 10351,
         9036, 12168, 10529, 15989,  9719, 15434, 10552, 11188, 13362,  9036,
        15805, 11300, 11846,  9146, 16691,  9181,  7397, 15806, 13480, 11342,
        17596,  9161, 19996,  9025, 25006, 18595,  9966, 12592, 10751, 11814,
         8711,  9046, 12450,  9117,  7377, 12521,     1])
output: tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -10

In [22]:
# train_dataset.input_ids[0]를 디코딩해보세요.
decoded_text = tokenizer.decode(train_dataset.input_ids[0])
print(decoded_text)

### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.</s>


In [ ]:
# output_dir="test": 공부하면서 중간중간 저장할 결과물(체크포인트)을 "test"라는 폴더에 담아두라는 뜻입니다.
# num_train_epochs=1: 전체 문제집(12,000개 데이터)을 딱 1번만 처음부터 끝까지 훑으며 공부하겠다는 설정입니다.
# per_device_train_batch_size=8: 한 번에 몇 문제를 풀고 채점할지 정합니다. 여기서는 8문제씩 묶어서 한꺼번에 학습하라는 뜻입니다. (이 숫자가 클수록 학습은 빠르지만 메모리를 많이 잡아먹습니다.)
# warmup_steps=5: 운동 전 준비운동처럼, 처음 5번의 학습 단계 동안은 공부 강도(학습률)를 아주 천천히 올리며 적응하라는 뜻입니다.
# prediction_loss_only=True: 학습 결과로 정답을 맞혔는지 세세한 데이터는 나중에 보고, 일단 "얼마나 틀렸는지(Loss)" 수치에만 집중해서 훈련하라는 옵션입니다.
# fp16 = True: 소수점 계산을 더 가볍게(16비트) 해서 학습 속도를 2배 이상 빠르게 올리는 마법의 스위치입니다. (단, Mac에서는 꺼두는 게 안전할 수 있습니다.)



training_args = transformers.TrainingArguments(
    output_dir="test",
    num_train_epochs=1,
    per_device_train_batch_size=1, # 맥북 메모리 안정성을 위해 4 권장
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,        # 👈 [강력 추천] 중간 계산값을 저장하지 않고 그때그때 다시 계산합니다

    
    per_device_eval_batch_size=1,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16=False,        # 👈 반드시 False로 끄세요! (맥북 MPS 충돌 방지)
    bf16=True,          # 👈 M4에서 강력 추천! 연산 속도를 비약적으로 올립니다.
)

# fully train
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

In [15]:
import torch
import gc
# 1. 파이썬 가비지 컬렉터 가동 (안 쓰는 메모리 수거)
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

gc.collect()
# 2. 맥북 MPS (메모리) 캐시 비우기
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
print("✨ 맥북 메모리 청소 완료!")

✨ 맥북 메모리 청소 완료!


In [ ]:
#로컬에서 ipynb로 돌리니 out of mps가 나서 파이썬 파일을 다로 만들어서 모델을 돌렸다...
trainer.train()
model.save_pretrained('models/output_1_SFT')

In [ ]:
# generator추론
generator = transformers.pipeline('text-generation', model='models/output_1_SFT', tokenizer=tokenizer)

# num_beams=4: 정답을 고를 때 한 가지만 생각하지 말고, 가장 가능성 있는 4가지 길을 동시에 검토해서 가장 매끄러운 답을 고르라는 뜻입니다. (신중함 증가)
# repetition_penalty=2.0: 똑같은 말이나 단어를 자꾸 반복하면 벌점을 줘서 중언부언하지 않게 막는 필터입니다.
# eos_token_id=375: 여기서는 줄바꿈(\n) 기호입니다. "대답하다가 엔터(줄바꿈)를 치면 답변이 끝난 걸로 간주하고 멈춰!"라고 가이드라인을 준 것입니다.
# max_new_tokens=64: 답변을 너무 길게 늘어지지 않게 최대 64단어(토큰)까지만 말하라는 제한입니다.
# do_sample=True: 매번 똑같은 답만 하지 말고 **약간의 창의성(주사위 굴리기)**을 발휘하라는 스위치입니다.

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=375, # \n
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

print(list_prompt)

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['### Instruction(명령어):\n불고기용 고기 한우에요?\n\n### Response(응답):', '### Instruction(명령어):\n리처드 닉슨이 43대 부통령직을 수행한 년도는?\n\n### Response(응답):', '### Instruction(명령어):\n시카고 오헤어 국제공항은 어디에 있어?\n\n### Response(응답):', '### Instruction(명령어):\n오늘 미세먼지 어때?\n\n### Response(응답):']


Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 AI 어시스턴트이기 때문에 고기를 먹을 수 없습니다. 하지만 일반적으로 불고기용 고기는 소고기, 돼지고기, 닭고기 등 다양한 요리에 사용됩니다. 따라서 어떤 종류의 고기를 원하시는지 알려주시면 더 정확한 답변을 드릴 수 있을 것 같습니다.了)了)

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨은 47대 부통령직을 수행했습니다. 불교에서는 "율리우스"라고도 불립니다. 불교에서는 '율리우스'라 불립니다. 따라서 리처드 닉슨이 41대 부통령직을 맡았던 년도는 정확히 알 수 없습니다. 불교에서는 '이카리우스'라고 불립니다. 따라서, 리처드 닉슨이 46대 부통령직을

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어?

### Response(응답):'저는 인공지능 어시스턴트이기 때문에 시카고에 대한 정보를 가지고 있지 않습니다. 하지만 시카고는 미국 캘리포니아주 로스앤젤레스에 위치해 있습니다.療港)이라고도 불립니다.療港이라고도 불립니다:\n\n시카고는 미국 캘리포니아주에 위치한 도시입니다.療港)이라고 불립니다.

### Instruction(명령어):
오늘 미세먼지 어때?

### Response(응답):'죄송합니다, 저는 인공지능 언어모델로써 미세먼지 상황을 파악할 수 없습니다. 미세먼지 상황이 어떤 것인지 자세히 설명해주시면 더 정확한 답변을 드릴 수 있을 것 같습니다. 감사합니다.增補)增補)은 미세먼지를 일으키는 원인 중 하나입니다.增補


In [19]:
# 맥북(MPS) 전용 메모리 비우기
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
print("✨ 맥북 MPS 메모리 청소 완료!")

✨ 맥북 MPS 메모리 청소 완료!


GPT-2를 Reward Model로 선택하는 이유는 다음과 같습니다.

1. 사전 학습(pretraining)의 이점
GPT-2는 대규모 텍스트 데이터에 대해 사전 학습되어 다양한 언어 패턴과 문맥 정보를 이미 학습한 상태입니다. 이를 통해, SFT 모델이 생성한 텍스트에 대해 미세하게 언어적, 문맥적 평가를 수행할 수 있습니다.

2. 아키텍처의 유사성 및 일관성
SFT 모델이 Transformer 기반이라면, GPT-2 역시 같은 Transformer 아키텍처를 사용합니다. 이로 인해 두 모델 간의 표현 공간이 어느 정도 일치할 가능성이 있으며, 텍스트의 흐름이나 품질을 평가하는 데 있어 더 자연스러운 비교가 가능해집니다.

3. 효율성과 구현 용이성
GPT-2는 오픈 소스 커뮤니티에서 널리 사용되고 검증된 모델로, 리소스나 구현 측면에서 안정적입니다. 따라서 복잡한 Reward Model을 처음부터 구축하기보다는, GPT-2를 기반으로 미세 조정하여 활용하는 것이 효율적입니다.

4. 자연스러운 텍스트 생성 및 평가
GPT-2는 이미 언어 생성 능력이 뛰어나므로, 생성된 텍스트의 문법적, 의미적 적합성을 평가할 때 신뢰성 있는 피드백을 제공할 수 있습니다.

In [6]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer

from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

import torch.nn as nn

import random

In [8]:
class GPTRM_custom(RewardModel):

    def __init__(self,
                 pretrained: Optional[str] = None,
                 config: Optional[GPT2Config] = None,
                 checkpoint: bool = False,
                 lora_rank: int = 0,
                 lora_train_bias: str = 'none',
                 tokenizer=None) -> None:
        # skt/kogpt2-base-v2         
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            model = GPT2Model(config)
        else:
            model = GPT2Model(GPT2Config())
        if checkpoint:
            model.gradient_checkpointing_enable()

        # 원래 GPT-2의 끝부분은 "수만 개의 단어 중 하나를 골라라!"라고 되어 있습니다.
        # 이걸 떼어내고 **"문장을 읽고 난 뒤, 딱 숫자 하나(점수)만 출력해라!"**라는 전용 채점 센서(Linear layer)를 
        # 새로 달아주는 핵심 코드입니다. 결과가 1인 이유는 점수가 숫자 '하나'이기 때문입니다.     

        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)

        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained


    def save_pretrained(self, dir):
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [9]:
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token='</s>', 
    eos_token='</s>', 
    unk_token='<unk>',  # 변경 (모르는 단어)
    pad_token='<pad>',  # 변경 (빈칸 채우기용 특수기호)
    mask_token='<mask>', # 추가 
    padding_side="right",
    model_max_length=512
)

# 이 부분은 마치 수술 전 소독된 수술대를 준비하는 것과 같습니다.
# 그냥 모델을 만들면 메모리 효율이 떨어지거나 장치(device) 설정이 꼬일 수 있습니다. 
# 그래서 NaiveStrategy라는 가이드를 따라 **"가장 깨끗하고 효율적인 상태"**에서 모델을 조립하기 시작합니다.
with NaiveStrategy().model_init_context():
        model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=tokenizer).to(device)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
# [{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?',
#   'completion_0': 'Allow me to answer your question. I know that you are curious about me.',
#   'completion_1': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.',
#   'completion_2': '라이언에게 말했다.',
#   'ranking': [2, 1, 0]},
#  {'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?',
#   'completion_0': '개포주공아파트는 다섯 단지로 이루어져 있습니다.',
#   'completion_1': '이날 목송에서 구글상위노',
#   'completion_2': '개포주공아파트는 총 27개 단지로 이루어져 있습니다.',
#   'ranking': [2, 0, 1]},
#  {'prompt': '김영삼의 후보 시절 지역표심을 겨냥한 발언을 문제삼은 후보는?',
#   'completion_0': 'The diameter of the Metallic domain is bigger than the Hyperonic domain.',
#   'completion_1': '이 질문은 조금 불분명합니다. 김영삼 대통령이 후보 시절에 어떤 발언을 했고, 누가 그 발언을 문제삼았는지에 따라 답이 다를 수 있습니다.\\n\\n만약 김영삼 대통령이 후보 시절에 지역표심을 겨냥한 발언을 했다는 가정하에, 그 발언을 문제삼은 후보가 누구였는지를 대답하자면, 그 답은 이화선 당시 민주당 대통령 후보가 될 것입니다. 1992년 총선 때, 김영삼 대선후보는 "집값이 오른 노량진역 부근의 부동산 가격은 세월호 폭침 후 \\\'강남 도시재생\\\' 일환으로 상승했다"는 발언을 했습니다. 하지만 이화선 후보는 이 발언을 "전국적으로 경제적 발전이 이루어지지 않은 지방민의 마음을 멀리해지려는 무례한 발언"이라고 비판하며 문제삼았습니다.\\n\\n하지만, 이 질문을 답변하는 데 있어서 보다 명확한 정보가 있으면 답변을 보완할 수 있습니다.',
#   'completion_2': '김영삼의 후보 시절에 지역표심을 겨냥한 발언은 대통령 당선 전까지 대한민국 정부가 추구하고 있는 민주주의 광범위하게 확립과 보수의 사상을 이어가는 데 있어 지역경제 발전과 공공서비스 신속 개선을 위해 합리적인 국가 정책에 따르는 방향성을 제시하고 있습니다.',
#   'ranking': [1, 2, 0]}]

# (Reward Model)에게 좋고 나쁜 답변을 구별하는 눈을 길러주기 위해, 데이터를 가공하는 과정"**입니다.

# 가장 중요한 포인트는 **"비교(Comparison)"**입니다. 인공지능에게 단순히 "이건 좋은 답이야"라고 가르치는 게 아니라, 
# **"A와 B 중에 A가 더 좋은 답이야!"**라고 가르치기 위해 데이터를 재배열하는 것입니다.


with open('/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

total_data_ranking2chosen = []
for tmp in list_data_dict:
    one_data_ranking2chosen = []

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_1']
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)

    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)



    total_data_ranking2chosen.extend(one_data_ranking2chosen)

print('before data num: %d'%(len(list_data_dict)))
print('after  data num: %d'%(len(total_data_ranking2chosen)))
print('data example: \n%s'%total_data_ranking2chosen[0])
print('data example: \n%s'%total_data_ranking2chosen[1])
print('data example: \n%s'%total_data_ranking2chosen[2])
print('data example: \n%s'%total_data_ranking2chosen[3])

before data num: 10220
after  data num: 30660
data example: 
{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?', 'chosen': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.', 'rejected': 'Allow me to answer your question. I know that you are curious about me.'}
data example: 
{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?', 'chosen': '라이언에게 말했다.', 'rejected': 'Allow me to answer your question. I know that you are curious about me.'}
data example: 
{'prompt': '번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?', 'chosen': '라이언에게 말했다.', 'rejected': '번디는 다양한 인터뷰자들과 뉴스홍보 담당자들과의 면담 때 밝혔다.'}
data example: 
{'prompt': '개포주공아파트는 몇 단지로 이루어져 있나?', 'chosen': '이날 목송에서 구글상위노', 'rejected': '개포주공아파트는 다섯 단지로 이루어져 있습니다.'}


In [19]:
class PairWiseLoss(nn.Module):

    def forward(self, chosen_reward: torch.Tensor, reject_reward: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(chosen_reward - reject_reward)
        # 잘했을 때: 확률이 1에 가까우면 log(1)은 0이 됩니다. 즉, "벌점(Loss)이 0점!" (잘했어!)
        # 못했을 때: 확률이 0에 가까우면(반대로 채점하면) 벌점이 무한대로 커집니다. (이놈! 똑바로 채점해!)
        log_probs = torch.log(probs)
        loss = -log_probs.mean()
        return loss

In [ ]:
# 잘못된 코드
# total_data_ranking2chosen = []

# for tmp in list_data_dict:
#      prompt = tmp['prompt']
#      ranking = tmp['ranking']

#       [1, 2, 0]
#      for index in range(1, len(ranking)):
#          n = ranking[0]
#          m = ranking[index]


#          data = {
#              'prompt': prompt,
#              'chosen': tmp['completion_{}'.format(n)],
#              'rejected': tmp['completion_{}'.format(m)]
#          }

#          total_data_ranking2chosen.append(data)



#고친코드
# total_data_ranking2chosen = []

# for tmp in list_data_dict:
#      prompt = tmp['prompt']
#      ranking = tmp['ranking']

#      # 1. 진짜 1등(순위가 0인 것)이 몇 번 답변인지 찾습니다.
#      winner_index = ranking.index(0) 

#      # 2. 1등을 제외한 나머지 답변들과 대결을 붙입니다.
#      for i in range(len(ranking)):
#          if i == winner_index: 
#              continue # 자기 자신과는 싸울 수 없으니까요!

#          data = {
#              'prompt': prompt,
#              'chosen': tmp[f'completion_{winner_index}'], # 진짜 1등 답변
#              'rejected': tmp[f'completion_{i}']            # 그 외의 답변
#          }
#          total_data_ranking2chosen.append(data)


In [20]:
import random
random.seed(230319)
random.shuffle(total_data_ranking2chosen)
print(total_data_ranking2chosen[45])

{'prompt': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은?', 'chosen': '유아인이 류승완 감독을 만나 영화 베테랑의 시나리오를 받았던 곳은 류승완의 사무실입니다.', 'rejected': '대구 영화사옥'}


In [21]:
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]

print(len(train_data))
print(len(eval_data))

train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)

1000
200


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [22]:
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])

######################################################################
## prompt ##
흑고래의 무게는 어느 정도야
######################################################################
## chosen ##
흑고래의 평균 몸무게는 약 25~40톤 정도이지만, 최대 몸무게는 50톤 이상에 이를 수 있습니다.
######################################################################
## rejected ##
흑고래의 무게는 매우 다양하게 달라집니다. 약 200kg에서 10톤까지 달라질 수 있습니다.


In [23]:
trainer = RewardModelTrainer(model=model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=4,
                             max_epochs=1)

In [24]:
trainer.fit(use_lora=0)

model.save_pretrained('models/output_2_RM')

Train epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Train step of epoch 0:   0%|          | 0/250 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

1. "질문"이 빠진 채로 "답변"만 채점하고 있습니다. (가장 결정적!)
보상 모델은 훈련할 때 [질문(Prompt)] + [답변(Completion)] 세트를 한 덩어리로 묶어서 배웠습니다.

모델의 생각: "어떤 질문에 대한 답변인지를 알아야 점수를 주지! 그냥 문장만 툭 던져주면 이건 대화가 아니잖아?"
현상: 질문 없이 답변만 들어오면 모델 입장에서는 **'갑자기 튀어나온 이상한 문장'**으로 인식해서 점수를 일단 깎고 시작합니다. (마치 시험 문제 없이 정답만 쓴 답안지를 채점하는 격입니다.)


2. SFT(말투 공부)를 건너뛰고 RM(채점 공부)을 했습니다.
아까 확인했듯이, 지금 

model
은 1단계 SFT 모델을 기반으로 하지 않고 원본 KoGPT2를 기반으로 만들어졌습니다.

원본 모델의 특징: "예의 바른 말투"나 "챗봇의 규칙"을 전혀 모릅니다.
문제: 말투의 기본기가 없는 모델에게 곧바로 "채점해봐!"라고 시켰기 때문에, 어떤 게 좋은 답변인지 판단하는 기준이 매우 불안정하고 좁습니다. (마치 한국어를 갓 배우기 시작한 외국인에게 논술 채점을 맡긴 것과 비슷합니다.)


3. 보상 점수(Reward Score)는 '절대 점수'가 아닙니다.
이 점수는 우리가 흔히 생각하는 0~100점 만점 수치가 아니라, 모델 내부의 **'수학적 확률값(Logit)'**입니다.

특징: 마이너스 수치 자체가 나오는 것은 지극히 정상입니다. 중요한 것은 "좋은 문장이 나쁜 문장보다 상대적으로 점수가 높은가?" 하는 것입니다.
결과: 지금 좋은 문장이 점수가 더 낮게 나오는 것은, 위 1, 2번 이유 때문에 모델이 완전히 **'혼란(Confusion)'**에 빠져 있다는 증거입니다.

In [33]:
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)

    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]

    print('input: %s\nreward score: %.1f'%(input_text, output_reward))

    return output_reward

input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)

input: 인공지능은 똥멍청이 입니다
reward score: -3.0


In [31]:
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'

output_reward = inference_RM(input_text=input_text)

input: 인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.
reward score: -3.2


In [32]:
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."

output_reward = inference_RM(input_text=input_text)

input: 인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다.
reward score: -3.1


In [30]:
# 비교 테스트용 문장
input_text_good = '인공지능은 사람에게 많은 도움을 줄 수 있는 유익한 도구입니다.'
output_reward_good = inference_RM(input_text=input_text_good)

input: 인공지능은 사람에게 많은 도움을 줄 수 있는 유익한 도구입니다.
reward score: -3.2


In [34]:
# 맥북(MPS) 전용 메모리 비우기
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
print("✨ 맥북 MPS 메모리 청소 완료!")

✨ 맥북 MPS 메모리 청소 완료!


## Proximal Policy Optimization

In [4]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

from copy import deepcopy

In [7]:
with NaiveStrategy().model_init_context():
    actor = GPTActor(pretrained='models/output_1_SFT', lora_rank=0).to(device)
    critic = GPTCritic(pretrained='models/output_2_RM', lora_rank=0).to(device)
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        "skt/kogpt2-base-v2",
        bos_token='</s>', 
        eos_token='</s>', 
        unk_token='<unk>',   # 모르는 단어 처리
        pad_token='<pad>',   # 빈칸 채우기용
        mask_token='<mask>',  # 마스킹용
        padding_side="right",
        model_max_length=512
    )
    initial_model = deepcopy(actor)
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(device)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [8]:
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [9]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

In [10]:
with open('/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.to(device) for k, v in batch.items()}

In [11]:
print(tokenize_fn('It takes something more than intelligence to act intelligently.'))

{'input_ids': tensor([[47311, 10448, 19008,  9792, 11780, 11308, 30190, 10929, 11849, 21663,
         44389,  9574, 13799,   458, 14308, 12778, 22469, 20938, 44696,   458,
         13799,   458, 14308, 12778, 11756, 18944,   389]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]], device='mps:0')}


In [12]:
len(list_prompt)

12000

In [13]:
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)

In [14]:
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)

actor.model.save_pretrained('models/output_3_PPO')

Episode [1/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

/Users/jamesyang/.pyenv/versions/3.12.2/envs/aipel/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Episode [2/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

...

Episode [9/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [10/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
def generation(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             num_return_sequences=1)
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)[0]
    print()
    print(output)
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    output = generation(input_text, actor)


### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):'저는 AI 어시스턴트이므로 현재 불고기용 쇠고기 한우의 시장 가격이 어떤 것인지 알 수 없습니다. 그러나, 보통 유통기한이 긴 한우의 경우 가격이 비싸다는 점은 알려져 있습니다. 하지만 일반적으로 유통기한이 오래된 한우의 경우 유통기간이 길수록 저렴한 경우가 있을 수 있습니다.  참조:\n\n또한, 일반적으로 냉동한 쇠고기를 사용하는 경우에도 매장판매를 위한 가격은 높은 수준일 수 있습니다. \n\n하지만, 유통기한은 유통기한 및 유통기한에 따라 가격이 다를 수 있으므로 정확한 정보를 얻기 위해서는 해당 부위의 쇠고기 판매처 또는 매장 직원에게 문의하시는 것이 가장 좋습니다.增  

### Instruction(명령어):
리처드 닉슨이 43대 부통령직을 수행한 년도는?

### Response(응답):'리처드 닉슨은 43대 부통령직을 수행한 연도는 433년입니다.: 그는 1950년대 미국의 부통령직을 역임하며 주로 군사 작전이나 군사 기밀을 담당하던 인물이었습니다., 舊治 賢治 등 다양한 분야에서 활동을 하였습니다.: 그는 1952년 미국 대통령 선거에서 당선되었습니다.: 그는 1940년대 후반에 부통령직에 출마하여 전쟁 영웅을 선발하였습니다.六),: 그는 1963년 대통령 선거에서 경쟁자인 앨런 케네디에게 패배하였습니다.?陟)顧顧顧顧顧察)顧顧顧顧察(顧察)이顧察)이며,罪,告察(顧顧察)高察,慧察),考察)에 능통하였고,能),伺察(考察)應考察,曹察)与察具無故 効果學技) 제물납,曹察,應報應: 效果察)에 능숙한 인물으로, 존경과 대우를 받는 인물이었습니다.了(宣祖 效

### Instruction(명령어):
시카고 오헤어 국제공항은 어디에 있어

### Response(응답):'시카고 오헤어 국제공항은 프랑스의 유럽 대륙 중 하나인 마추스 국제공항입니다.香土川島港)에 위치해 있으며, 약 16,000명 이상의 탑승객들에게 항공기와 항공권 등의 서비스를 제공하고 있습니다. 

: 